# MeAJOR SVM Fine-tuning
This notebook runs a grid of Linear SVM models wrapped with calibration and evaluates different `C` values, word n-gram ranges, and character n-gram ranges.

In [1]:
# Standard imports and project setup
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))
print('Repository root:', repo_root)

Repository root: C:\Users\brian\phishing-investigator


In [2]:
from src.ingestion.major_loader import load_major_training_dataset

DATASET_PATH = repo_root / 'data' / 'raw' / 'meajor_cleaned_preprocessed.csv'
print('Dataset path:', DATASET_PATH)
data = load_major_training_dataset(str(DATASET_PATH))
print('Loaded rows:', len(data))
X = data['text']
y = data['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train/Test sizes:', len(X_train), len(X_test))

Dataset path: C:\Users\brian\phishing-investigator\data\raw\meajor_cleaned_preprocessed.csv
Dropped 1 rows with missing labels
Loaded rows: 108684
Train/Test sizes: 86947 21737


In [3]:
# Grid settings
C_values = [0.1, 0.25, 0.5, 1.0, 1.5, 2.0, 2.5]
word_ngrams = [(1,2), (1,3)]
char_ngrams = [(3,5), (4,6)]

results = []

def evaluate_pipeline(pipe, X_train, X_test, y_train, y_test):
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = None
    try:
        probs = pipe.predict_proba(X_test)[:,1]
    except Exception:
        probs = None
    return {
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0),
        'probs': probs,
    }

# Iterate grid
for C in C_values:
    for w_ng in word_ngrams:
        for c_ng in char_ngrams:
            print(f'Testing C={C}, word_ng={w_ng}, char_ng={c_ng}')
            # Build combined vectorizer: word + char union via FeatureUnion isn't used here; instead, test word-only and char-only separately and combined by concatenation via pipeline
            # We'll create a Tfidf vectorizer that uses both word and char ngrams by using analyzer='char' and analyzer='word' separately and concatenating with ColumnTransformer-like approach using scikit-learn's FeatureUnion is better, but for simplicity, we'll try word-only and char-only and a combined string (word + char markers)
            # Word-level pipeline
            word_pipe = Pipeline([
                ('tfidf', TfidfVectorizer(ngram_range=w_ng, analyzer='word', max_features=20000)),
                ('clf', CalibratedClassifierCV(estimator=LinearSVC(C=C, max_iter=20000), cv=3))
            ])
            res_word = evaluate_pipeline(word_pipe, X_train, X_test, y_train, y_test)
            res_word.update({'C': C, 'word_ng': w_ng, 'char_ng': None, 'type': 'word'})
            results.append(res_word)

            # Char-level pipeline
            char_pipe = Pipeline([
                ('tfidf', TfidfVectorizer(ngram_range=c_ng, analyzer='char', max_features=20000)),
                ('clf', CalibratedClassifierCV(estimator=LinearSVC(C=C, max_iter=20000), cv=3))
            ])
            res_char = evaluate_pipeline(char_pipe, X_train, X_test, y_train, y_test)
            res_char.update({'C': C, 'word_ng': None, 'char_ng': c_ng, 'type': 'char'})
            results.append(res_char)

            # Combined: simple approach concatenating word + char by generating features separately and hstacking - use FeatureUnion for proper approach, but we'll approximate by joining both representations into one large vector via a ColumnTransformer and FeatureUnion could be added. For brevity we skip combined here.

# Summarize results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('f1', ascending=False).reset_index(drop=True)
display(results_df.head(20))

# Save results to CSV for later analysis
results_df.to_csv(repo_root / 'experiments' / 'svm_finetune_results.csv', index=False)
print('Saved results to experiments/svm_finetune_results.csv')

Testing C=0.1, word_ng=(1, 2), char_ng=(3, 5)
Testing C=0.1, word_ng=(1, 2), char_ng=(4, 6)
Testing C=0.1, word_ng=(1, 3), char_ng=(3, 5)
Testing C=0.1, word_ng=(1, 3), char_ng=(4, 6)
Testing C=0.25, word_ng=(1, 2), char_ng=(3, 5)
Testing C=0.25, word_ng=(1, 2), char_ng=(4, 6)
Testing C=0.25, word_ng=(1, 3), char_ng=(3, 5)
Testing C=0.25, word_ng=(1, 3), char_ng=(4, 6)
Testing C=0.5, word_ng=(1, 2), char_ng=(3, 5)
Testing C=0.5, word_ng=(1, 2), char_ng=(4, 6)
Testing C=0.5, word_ng=(1, 3), char_ng=(3, 5)
Testing C=0.5, word_ng=(1, 3), char_ng=(4, 6)
Testing C=1.0, word_ng=(1, 2), char_ng=(3, 5)
Testing C=1.0, word_ng=(1, 2), char_ng=(4, 6)
Testing C=1.0, word_ng=(1, 3), char_ng=(3, 5)
Testing C=1.0, word_ng=(1, 3), char_ng=(4, 6)
Testing C=1.5, word_ng=(1, 2), char_ng=(3, 5)
Testing C=1.5, word_ng=(1, 2), char_ng=(4, 6)
Testing C=1.5, word_ng=(1, 3), char_ng=(3, 5)
Testing C=1.5, word_ng=(1, 3), char_ng=(4, 6)
Testing C=2.0, word_ng=(1, 2), char_ng=(3, 5)
Testing C=2.0, word_ng=(1, 2),

,accuracy,precision,recall,f1,probs,C,word_ng,char_ng,type
0,0.986613,0.983697,0.986052,0.984873,"[8.113397610002107e-05, 0.9997538869346573, 0....",2.0,"(1, 3)",None,word
1,0.986613,0.983697,0.986052,0.984873,"[8.113397610002107e-05, 0.9997538869346573, 0....",2.0,"(1, 3)",None,word
2,0.986567,0.983695,0.985948,0.984820,"[0.00011007041209082449, 0.9990549548422217, 0...",2.5,"(1, 2)",None,word
3,0.986567,0.983695,0.985948,0.984820,"[0.00011007041209082449, 0.9990549548422217, 0...",2.5,"(1, 2)",None,word
4,0.986521,0.983593,0.985948,0.984769,"[0.00011097748571260193, 0.9990169167264397, 0...",2.0,"(1, 2)",None,word
5,0.986521,0.983593,0.985948,0.984769,"[0.00011097748571260193, 0.9990169167264397, 0...",2.0,"(1, 2)",None,word
6,0.986475,0.983290,0.986156,0.984721,"[8.869822698483259e-05, 0.9997236945896019, 0....",1.5,"(1, 3)",None,word
7,0.986475,0.983290,0.986156,0.984721,"[8.869822698483259e-05, 0.9997236945896019, 0....",1.5,"(1, 3)",None,word
8,0.986475,0.983491,0.985948,0.984718,"[0.00015174676288926998, 0.9988615971752823, 0...",1.0,"(1, 2)",None,word
9,0.986475,0.983491,0.985948,0.984718,"[0.00015174676288926998, 0.9988615971752823, 0...",1.0,"(1, 2)",None,word


Saved results to experiments/svm_finetune_results.csv
